# 키워드 연관 분석

- 대응 코드: `association.py`
- 문장 내 공출현 표본과 Ochiai 간선을 먼저 확인.
- 전체 네트워크 및 그림 생성은 선택 실행.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
LANG = "kr"
RUN_REPORT = False
PREVIEW_ROWS = 10

In [ ]:
from stage3_association import association as step
if LANG not in cfg.LANGS:
    raise ValueError(f"LANG은 {cfg.LANGS} 중 하나여야 합니다.")
merged, docs = load_enriched(LANG)
sentence_table = step.build_sentence_concept_table(merged)
print(f"문장·개념 행 {len(sentence_table):,}건")
display(sentence_table.head(PREVIEW_ROWS))

In [ ]:
category_of = step._category_of_map(merged)
edges = step.compute_edges(sentence_table, category_of)
print(f"연관 간선 {len(edges):,}개")
display(edges.sort_values("ochiai", ascending=False).head(PREVIEW_ROWS))

In [ ]:
if RUN_REPORT:
    step.build_stage3_report(LANG)
else:
    print("전체 연관 분석·그림 생성은 RUN_REPORT=True로 선택하세요.")

In [ ]:
out_dir = cfg.OUTPUT_DIR / LANG / "단계3_연관분석"
show_csv(out_dir / f"concept_edges_{LANG}.csv", PREVIEW_ROWS)
show_csv(out_dir / f"edge_samples_{LANG}.csv", PREVIEW_ROWS)